# GigaPath: Adenoma vs Carcinoma (patch embeddings → slide classifier)


## 0. Install/upgrade deps
필요하면 실행: torch/timm/huggingface-hub/openslide/scikit-image/pillow/pyarrow/pandas/scikit-learn


In [1]:
!pip install -U 'torch>=2.1' timm huggingface-hub pillow openslide-python scikit-image pyarrow pandas scikit-learn tqdm


## 1. Auth & paths
HF_TOKEN은 환경변수(HF_TOKEN/HUGGINGFACE_HUB_TOKEN) 또는 .env(./, ../, ../../)에서 로드합니다.
데이터: PoC/v1/mammary_adenoma_vs_adenocarcinoma_curated.parquet 기준으로 각 클래스 20개 슬라이드 샘플링.


In [2]:

import os
from pathlib import Path
from huggingface_hub import login, HfApi
import pandas as pd

def _load_hf_token():
    token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')
    if token:
        os.environ.setdefault('HUGGINGFACE_HUB_TOKEN', token)
        return token
    candidates = [Path('.env'), Path('../.env'), Path('../../.env')]
    for path in candidates:
        if path.exists():
            for line in path.read_text().splitlines():
                if not line or line.strip().startswith('#') or '=' not in line:
                    continue
                k, v = line.split('=', 1)
                if k.strip() in ('HF_TOKEN', 'HUGGINGFACE_HUB_TOKEN'):
                    token = v.strip()
                    os.environ['HF_TOKEN'] = token
                    os.environ.setdefault('HUGGINGFACE_HUB_TOKEN', token)
                    return token
    raise RuntimeError('HF_TOKEN을 환경변수 또는 .env에 설정하세요 (gated 모델 접근 필요).')

HF_TOKEN = _load_hf_token()
login(token=HF_TOKEN, add_to_git_credential=False)

PARQUET_PATH = Path('/Users/curv/Repos/GC-Pathology/PoC/v1/mammary_adenoma_vs_adenocarcinoma_curated.parquet')
DATA_ROOT = Path('/Users/curv/Repos/GC-Pathology/Data/2023')
EMBED_ROOT = Path('PoC/v1/output_embeddings')
TILE_ROOT = Path('PoC/v1/output_tiles')

if not PARQUET_PATH.exists():
    raise FileNotFoundError(PARQUET_PATH)
df = pd.read_parquet(PARQUET_PATH)
print('Loaded parquet rows:', len(df))


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loaded parquet rows: 48689


## 2. 샘플 선택 (각 클래스 20개 슬라이드)


In [3]:

import random
label_norm = df['label'].astype(str).str.lower().str.strip()
valid = {'mammary_adenoma': 0, 'mammary_adenocarcinoma': 1}
filtered = df.loc[label_norm.isin(valid.keys())].copy()
filtered['label'] = label_norm.loc[filtered.index].map(valid)
print('클래스별:', filtered['label'].value_counts().to_dict())
need_per_class = 20
samples = []
for lbl, val in valid.items():
    rows = filtered[filtered['label'] == val].sample(frac=1, random_state=42).to_dict('records')
    picked = []
    for row in rows:
        slide_dir = DATA_ROOT / str(row['FOLDER']).strip()
        file_names = [fn.strip() for fn in str(row['FILE_NAME']).split('|') if fn.strip()]
        for fn in file_names:
            path = slide_dir / fn
            if path.exists():
                picked.append({'slide_id': f"{row['INSP_RQST_NO']}:{fn}", 'path': path, 'label': val})
                break
        if len(picked) >= need_per_class:
            break
    print(f'label {lbl} picked {len(picked)}')
    samples.extend(picked)
if len(samples) < 2 * need_per_class:
    raise RuntimeError('슬라이드 수가 부족합니다. parquet/경로를 확인하세요.')
print('총 선택 슬라이드:', len(samples))
for rec in samples[:5]:
    print(rec['slide_id'], rec['path'])


클래스별: {0: 4346, 1: 3489}
label mammary_adenoma picked 20
label mammary_adenocarcinoma picked 20
총 선택 슬라이드: 40
20231023-142-0010:S23-06848#1###1.svs /Users/curv/Repos/GC-Pathology/Data/2023/S23-06848/S23-06848#1###1.svs
20230523-148-0004:S23-02954#1###5.svs /Users/curv/Repos/GC-Pathology/Data/2023/S23-02954/S23-02954#1###5.svs
20230330-138-0005:S23-01832#1###8.svs /Users/curv/Repos/GC-Pathology/Data/2023/S23-01832/S23-01832#1###8.svs
20230913-147-0001:S23-05668#1#A##8.svs /Users/curv/Repos/GC-Pathology/Data/2023/S23-05668/S23-05668#1#A##8.svs
20231130-122-0002:S23-08214#1###0.svs /Users/curv/Repos/GC-Pathology/Data/2023/S23-08214/S23-08214#1###0.svs


## 3. 모델 로드 (timm hf_hub: prov-gigapath/prov-gigapath)


In [4]:

import torch
import timm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_name = 'hf_hub:prov-gigapath/prov-gigapath'
model = timm.create_model(model_name, pretrained=True).to(device)
model.eval()
print('Loaded model on', device)


Loaded model on cpu


## 4. 타일링 + 임베딩 추출 & Parquet 저장
각 슬라이드당 최대 200 타일, tissue mask 적용. Parquet: slide_id, label, x, y, level, tile_path, feature(list).


In [5]:

import numpy as np
import shutil
import openslide
from skimage import color, filters
from PIL import Image, ImageFile, PngImagePlugin, UnidentifiedImageError
from tqdm import tqdm
import pyarrow as pa
import pyarrow.parquet as pq
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform

PngImagePlugin.MAX_TEXT_CHUNK = None
ImageFile.LOAD_TRUNCATED_IMAGES = True

PATCH_SIZE = 256
MIN_TISSUE_RATIO = 0.6
LEVEL = 0
MAX_TILES_PER_SLIDE = 200

cfg = resolve_data_config({}, model=model)
transform = create_transform(**cfg)

def tissue_mask(np_rgb: np.ndarray) -> np.ndarray:
    hsv = color.rgb2hsv(np_rgb)
    saturation = hsv[:, :, 1]
    thresh = filters.threshold_otsu(saturation)
    return saturation > thresh

def iter_tiles(slide: openslide.OpenSlide, stride: int = PATCH_SIZE):
    width, height = slide.level_dimensions[LEVEL]
    for y in range(0, height, stride):
        for x in range(0, width, stride):
            region = slide.read_region((x, y), LEVEL, (PATCH_SIZE, PATCH_SIZE)).convert('RGB')
            arr = np.array(region)
            mask = tissue_mask(arr)
            if mask.mean() < MIN_TISSUE_RATIO:
                continue
            yield x, y, region

def export_tiles(slide_path: Path, out_dir: Path, max_tiles: int = MAX_TILES_PER_SLIDE):
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    slide = openslide.OpenSlide(str(slide_path))
    meta = []
    for tile_idx, (x, y, tile_img) in enumerate(iter_tiles(slide)):
        tile_name = f"tile_{tile_idx:06d}_x{x}_y{y}.png"
        tile_path = out_dir / tile_name
        tile_img.save(tile_path, format='PNG', icc_profile=None)
        meta.append({'tile_path': str(tile_path), 'x': x, 'y': y, 'level': LEVEL})
        if len(meta) >= max_tiles:
            break
    slide.close()
    return meta

def embed_slide(rec):
    slide_path = rec['path']
    slide_id = rec['slide_id']
    label = int(rec['label'])
    tile_dir = TILE_ROOT / slide_path.stem
    print(f"[slide start] {slide_id} -> {tile_dir}")
    meta = export_tiles(slide_path, tile_dir, max_tiles=MAX_TILES_PER_SLIDE)
    print(f"[tiling done] {slide_id}: kept {len(meta)} tiles")
    if not meta:
        print(f'No tiles kept for {slide_id}')
        return
    records = []
    batch = []
    paths_batch = []
    bs = 16
    skipped = 0
    for idx, m in enumerate(meta):
        try:
            img = Image.open(m['tile_path']).convert('RGB')
        except UnidentifiedImageError as e:
            skipped += 1
            if skipped <= 5:
                print(f"[skip] unreadable tile: {m['tile_path']} ({e})")
            continue
        except Exception as e:
            skipped += 1
            if skipped <= 5:
                print(f"[skip] other error: {m['tile_path']} ({e})")
            continue
        batch.append(transform(img))
        paths_batch.append(m)
        if len(batch) == bs or idx == len(meta) - 1:
            x = torch.stack(batch).to(device)
            with torch.no_grad():
                feats = model.forward_features(x)
                if isinstance(feats, (list, tuple)):
                    feats = feats[-1]
                if feats.ndim == 4:
                    feats = feats.mean(dim=(2, 3))
            feats = feats.cpu().numpy().astype('float32')
            for f, pm in zip(feats, paths_batch):
                records.append({
                    'slide_id': slide_id,
                    'label': label,
                    'x': pm['x'],
                    'y': pm['y'],
                    'level': pm['level'],
                    'tile_path': pm['tile_path'],
                    'feature': f.tolist(),
                })
            batch, paths_batch = [], []
    table = pa.Table.from_pylist(records)
    EMBED_ROOT.mkdir(parents=True, exist_ok=True)
    out_path = EMBED_ROOT / f"{slide_path.stem}.parquet"
    pq.write_table(table, out_path, compression='zstd')
    print(f"[embed saved] {slide_id}: {len(records)} feats, skipped {skipped} -> {out_path}")

for rec in samples:
    embed_slide(rec)


[slide start] 20231023-142-0010:S23-06848#1###1.svs -> PoC/v1/output_tiles/S23-06848#1###1
[tiling done] 20231023-142-0010:S23-06848#1###1.svs: kept 200 tiles
[embed saved] 20231023-142-0010:S23-06848#1###1.svs: 200 feats, skipped 0 -> PoC/v1/output_embeddings/S23-06848#1###1.parquet
[slide start] 20230523-148-0004:S23-02954#1###5.svs -> PoC/v1/output_tiles/S23-02954#1###5
[tiling done] 20230523-148-0004:S23-02954#1###5.svs: kept 200 tiles
[embed saved] 20230523-148-0004:S23-02954#1###5.svs: 200 feats, skipped 0 -> PoC/v1/output_embeddings/S23-02954#1###5.parquet
[slide start] 20230330-138-0005:S23-01832#1###8.svs -> PoC/v1/output_tiles/S23-01832#1###8
[tiling done] 20230330-138-0005:S23-01832#1###8.svs: kept 200 tiles
[embed saved] 20230330-138-0005:S23-01832#1###8.svs: 200 feats, skipped 0 -> PoC/v1/output_embeddings/S23-01832#1###8.parquet
[slide start] 20230913-147-0001:S23-05668#1#A##8.svs -> PoC/v1/output_tiles/S23-05668#1#A##8
[tiling done] 20230913-147-0001:S23-05668#1#A##8.svs

## 5. 임베딩 로드 → 슬라이드 MIL 분류기 학습/검증
Attention MIL로 타일을 집계 후 Linear classifier 학습.


In [3]:

import glob
import numpy as np
import pyarrow.parquet as pq
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim

# load slide-level bags
parquet_files = sorted(glob.glob(str(EMBED_ROOT / '*.parquet')))
if not parquet_files:
    raise RuntimeError('임베딩 parquet가 없습니다. 이전 셀을 실행하세요.')
slides = []
for fp in parquet_files:
    tbl = pq.read_table(fp)
    feats = np.vstack(tbl['feature'].to_pylist()).astype('float32')  # (n_tiles, D)
    labels = tbl['label'].to_numpy()
    label = int(np.unique(labels)[0]) if len(set(labels)) else int(labels[0])
    slides.append({'features': feats, 'label': label, 'name': Path(fp).stem})
print('loaded slides:', len(slides), 'dim:', slides[0]['features'].shape[1])

# train/val split by slide
labels = np.array([s['label'] for s in slides])
train_idx, val_idx = train_test_split(np.arange(len(slides)), test_size=0.2, stratify=labels, random_state=42)
train_slides = [slides[i] for i in train_idx]
val_slides = [slides[i] for i in val_idx]
print('train slides:', len(train_slides), 'val slides:', len(val_slides))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
D = slides[0]['features'].shape[1]

class AttnMIL(nn.Module):
    def __init__(self, in_dim, hidden=256):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.tanh = nn.Tanh()
        self.attn = nn.Linear(hidden, 1)
        self.classifier = nn.Linear(in_dim, 1)
    def forward(self, x):  # x: (n_tiles, D)
        h = self.tanh(self.fc1(x))  # (n, hidden)
        a = self.attn(h).squeeze(-1)  # (n)
        a = torch.softmax(a, dim=0)
        m = torch.sum(a.unsqueeze(-1) * x, dim=0)  # (D)
        logit = self.classifier(m)
        return logit, a, m

model_head = AttnMIL(D, hidden=256).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_head.parameters(), lr=1e-3)
epochs = 10

for epoch in range(1, epochs + 1):
    model_head.train()
    train_losses = []
    for s in train_slides:
        x = torch.tensor(s['features'], device=device)
        y = torch.tensor(float(s['label']), device=device)
        optimizer.zero_grad()
        logit, attn, _ = model_head(x)
        loss = criterion(logit.view(()), y)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
    # validation
    model_head.eval()
    val_logits = []
    val_labels = []
    with torch.no_grad():
        for s in val_slides:
            x = torch.tensor(s['features'], device=device)
            logit, attn, _ = model_head(x)
            val_logits.append(logit.item())
            val_labels.append(s['label'])
    if val_logits:
        val_prob = torch.sigmoid(torch.tensor(val_logits))
        val_pred = (val_prob >= 0.5).long().numpy()
        val_acc = (val_pred == np.array(val_labels)).mean()
    else:
        val_acc = float('nan')
    print(f'Epoch {epoch}: train_loss={np.mean(train_losses):.4f} val_acc={val_acc:.3f}')


loaded slides: 40 dim: 1536
train slides: 32 val slides: 8
Epoch 1: train_loss=0.8037 val_acc=0.500
Epoch 2: train_loss=0.6506 val_acc=0.250
Epoch 3: train_loss=0.5663 val_acc=0.750
Epoch 4: train_loss=0.4144 val_acc=0.375
Epoch 5: train_loss=0.4106 val_acc=0.750
Epoch 6: train_loss=0.2422 val_acc=0.625
Epoch 7: train_loss=0.1537 val_acc=0.750
Epoch 8: train_loss=0.2862 val_acc=0.500
Epoch 9: train_loss=0.1044 val_acc=0.500
Epoch 10: train_loss=0.0789 val_acc=0.750
